# 5-Year Pipeline: Part 5 - Strategy Model Inference & Evaluation
This notebook evaluates the Strategy Models (Base and Refined) and logs the results to `models/model_performance_log.json`.

In [1]:
import joblib
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
import json
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

def update_model_log(model_name, metrics, features_count, removed_features=0):
    log_file = 'models/model_performance_log.json'
    os.makedirs('models', exist_ok=True)
    log_data = {}
    if os.path.exists(log_file):
        with open(log_file, 'r') as f:
            try: log_data = json.load(f)
            except: pass
    log_data[model_name] = {
        "date_trained": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "metrics": metrics,
        "features_count": features_count,
        "removed_features": removed_features
    }
    with open(log_file, 'w') as f:
        json.dump(log_data, f, indent=4)
    print(f"✅ Updated JSON log for: {model_name}")

print("Loading Test Data...")
df = pd.read_csv('5_year_final_data.csv')
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df['date'] = pd.to_datetime(df['date'])

df_signals = df[df['base_signal'] == True]
df_signals = df_signals[~df_signals['outcome'].isin(['No_Trigger', 'Pending', 'Timeout', np.nan])]
df_signals = df_signals[df_signals['risk'] > 0]
df_signals['max_r_achieved'] = df_signals['mfe'] / df_signals['risk']
df_signals = df_signals.dropna(subset=['max_r_achieved'])
df_signals = df_signals.sort_values('date').reset_index(drop=True)

n = len(df_signals)
test_df = df_signals.iloc[int(n * 0.80):].copy()

with open('base_features.txt', 'r') as f: base_features = [line.strip() for line in f.readlines()]
with open('final_features.txt', 'r') as f: refined_features = [line.strip() for line in f.readlines()]

Loading Test Data...


In [2]:
print("\n--- EVALUATING BASE MODEL ---")
base_model = joblib.load('models/strategy_base_model.pkl')
base_scaler = joblib.load('models/strategy_base_scaler.pkl')
base_imputer = joblib.load('models/strategy_base_imputer.pkl')

X_test_base = test_df[base_features]
X_test_s_base = base_scaler.transform(base_imputer.transform(X_test_base))
preds_base = base_model.predict(X_test_s_base)

test_df['predicted_r_base'] = preds_base
test_df['actual_is_success'] = test_df['max_r_achieved'] >= 1.0
test_df['predicted_is_success_base'] = test_df['predicted_r_base'] >= 1.0

acc_base = (test_df['actual_is_success'] == test_df['predicted_is_success_base']).mean() * 100
mae_base = mean_absolute_error(test_df['max_r_achieved'], preds_base)

print(f"Base Model Accuracy: {acc_base:.1f}% | MAE: {mae_base:.2f} R")
update_model_log("Strategy_Base", {"Accuracy": f"{acc_base:.1f}%", "MAE": round(mae_base,3)}, len(base_features), 0)


--- EVALUATING BASE MODEL ---
Base Model Accuracy: 44.6% | MAE: 0.89 R
✅ Updated JSON log for: Strategy_Base


In [3]:
print("\n--- EVALUATING REFINED MODEL ---")
refined_model = joblib.load('models/5_year_xgboost_model.pkl')
refined_scaler = joblib.load('models/5_year_scaler.pkl')
refined_imputer = joblib.load('models/5_year_imputer.pkl')

X_test_ref = test_df[refined_features]
X_test_s_ref = refined_scaler.transform(refined_imputer.transform(X_test_ref))
preds_ref = refined_model.predict(X_test_s_ref)

test_df['predicted_r_ref'] = preds_ref
test_df['predicted_is_success_ref'] = test_df['predicted_r_ref'] >= 1.0

acc_ref = (test_df['actual_is_success'] == test_df['predicted_is_success_ref']).mean() * 100
mae_ref = mean_absolute_error(test_df['max_r_achieved'], preds_ref)
actual_successes = test_df[test_df['actual_is_success'] == True]
successes_caught = actual_successes['predicted_is_success_ref'].sum()
total_actual_successes = len(actual_successes)
recall = (successes_caught / total_actual_successes) * 100 if total_actual_successes > 0 else 0

print(f"Refined Model Accuracy: {acc_ref:.1f}% | MAE: {mae_ref:.2f} R")
print(f"Caught {successes_caught} out of {total_actual_successes} successful trades ({recall:.1f}%).")

update_model_log("Strategy_Refined", {"Accuracy": f"{acc_ref:.1f}%", "Win_Rate_Recall": f"{recall:.1f}%", "MAE": round(mae_ref,3)}, len(refined_features), len(base_features) - len(refined_features))


--- EVALUATING REFINED MODEL ---
Refined Model Accuracy: 43.9% | MAE: 0.89 R
Caught 306 out of 396 successful trades (77.3%).
✅ Updated JSON log for: Strategy_Refined


In [4]:
print("\n--- EVALUATING INDIVIDUAL STRATEGY MODELS ---")
individual_dir = 'models/individual_model/strategy'
individual_logs = {}
acc_list = []
mae_list = []

for sym in test_df['symbol'].unique():
    model_path = f"{individual_dir}/{sym}_model.pkl"
    if os.path.exists(model_path):
        sym_df = test_df[test_df['symbol'] == sym]
        if len(sym_df) == 0: continue
        
        try:
            model = joblib.load(model_path)
            scaler = joblib.load(f"{individual_dir}/{sym}_scaler.pkl")
            imputer = joblib.load(f"{individual_dir}/{sym}_imputer.pkl")
            
            X_test_sym = sym_df[refined_features]
            X_test_s = scaler.transform(imputer.transform(X_test_sym))
            preds = model.predict(X_test_s)
            
            mae = mean_absolute_error(sym_df['max_r_achieved'], preds)
            pred_success = preds >= 1.0
            actual_success = sym_df['max_r_achieved'] >= 1.0
            acc = (actual_success == pred_success).mean() * 100
            
            acc_list.append(acc)
            mae_list.append(mae)
            individual_logs[sym] = {"Accuracy": f"{acc:.1f}%", "MAE": round(mae, 3)}
        except Exception as e:
            pass

if acc_list:
    avg_acc = np.mean(acc_list)
    avg_mae = np.mean(mae_list)
    print(f"Individual Strategy Models (Avg): Accuracy {avg_acc:.1f}% | MAE {avg_mae:.2f} R")
    update_model_log("Individual_Strategy_Avg", {"Accuracy": f"{avg_acc:.1f}%", "MAE": round(avg_mae,3)}, len(refined_features))
    
    with open('models/individual_strategy_log.json', 'w') as f:
        json.dump(individual_logs, f, indent=4)
    print("✅ Saved individual strategy logs to models/individual_strategy_log.json")




--- EVALUATING INDIVIDUAL STRATEGY MODELS ---
